### **Tic-Tac-Toe — From Random Play to Policy Gradient to Model-Based Planning**

Practicals **PL7** (environment, features, random agents), **PL8** (REINFORCE with baseline), and **PL9** (Monte Carlo Tree Search).


In [ ]:
import sys, os, math, time
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import matplotlib.pyplot as plt

from AR1.envs.tictactoe import TicTacToeEnv, _winner
from AR1.features.tictactoe import encode_state, STATE_FEATURE_DIM
from AR1.experiments.tictactoe import play_game, play_game_vs_human
from AR1.agents.control.reinforce import ReinforceAgent
from AR1.experiments.reinforce_tictactoe import (
    train, make_reinforce_policy, evaluate_vs_random, _play_silent,
)
from AR1.agents.planning.mcts import MCTSAgent
from AR1.experiments.mcts_tictactoe import (
    make_mcts_policy,
    evaluate_vs_random as mcts_eval_vs_random,
    evaluate_mcts_vs_reinforce,
    evaluate_mcts_vs_mcts,
)
from AR1.policies.tictactoe import random_action as random_policy


---
#### **1. Two random agents playing (PL7)**

Before any learning, let's watch two completely random policies play a game.


In [ ]:
env = TicTacToeEnv()
play_game(env, random_policy, random_policy)


In [ ]:
# Statistics over 1000 random games
n = 1_000
outcomes = {1: 0, -1: 0, 0: 0}
for _ in range(n):
    state = env.reset()
    while not env.is_terminal(state):
        action = random_policy(env, state)
        state, _, _ = env.step(action)
    outcomes[_winner(state)] += 1

print(f"Over {n} random vs random games:")
print(f"  X wins: {outcomes[1]/n:.1%}  |  O wins: {outcomes[-1]/n:.1%}  |  Draws: {outcomes[0]/n:.1%}")


---
#### **2. The 27-dimensional feature vector (PL7)**

Each board state is encoded **from the current player's perspective**:

| Cell content | Encoding |
|---|---|
| My piece     | `[1, 0, 0]` |
| Opponent     | `[0, 1, 0]` |
| Empty        | `[0, 0, 1]` |

9 cells × 3 dims = **27 features**.

Because the encoding is relative, the same policy weights work for both X and O — essential for self-play.


In [ ]:
# Example board after three moves
#   X | . | .
#   . | O | .
#   . | . | X
sample_board = (1, 0, 0, 0, -1, 0, 0, 0, 1)
current_player = -1  # O's turn

phi = encode_state(sample_board, current_player)

fig, axes = plt.subplots(1, 3, figsize=(12, 2.5))
titles = ["my piece (dim 0,3,6,...)", "opponent (dim 1,4,7,...)", "empty (dim 2,5,8,...)"]
for ax, offset, title in zip(axes, range(3), titles):
    ax.imshow(phi[offset::3].reshape(3, 3), vmin=0, vmax=1, cmap="Greys")
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle(f"encode_state for O's perspective  (dim = {STATE_FEATURE_DIM})", y=1.05)
plt.tight_layout()
plt.show()


---
#### **3. REINFORCE algorithm (PL8)**

**Linear softmax policy** ($\theta \in \mathbb{R}^{9 \times 27}$):

$$\pi(a|s) = \frac{e^{\theta_a \cdot \phi(s)}}{\sum_{b \in \mathcal{A}(s)} e^{\theta_b \cdot \phi(s)}}$$

**Self-play**: X and O share the same $\theta$. Trajectories are collected separately per player; the loser's final step is re-labelled with $r = -1$ (the env emits $0$ on a loss, so REINFORCE would otherwise have no signal to push away from losing moves).

**Baseline** (Sutton & Barto, Sec. 13.4) reduces gradient variance without bias:

$$\delta_t = G_t - V(s_t) \qquad\qquad V(s) = w \cdot \phi(s)$$
$$\theta \mathrel{+}= \alpha \gamma^t\, \delta_t\, \nabla_\theta \log \pi(a_t|s_t)$$


---
#### **4. Training**


In [ ]:
SEED               = 42
NUM_EPISODES       = 50_000
ALPHA              = 0.005
GAMMA              = 1.0
ENTROPY_BETA       = 0.01
RANDOM_OPP_FRAC    = 0.3
EVAL_EVERY         = 2_000
EVAL_GAMES         = 500

env   = TicTacToeEnv()
agent = ReinforceAgent(
    alpha=ALPHA, gamma=GAMMA, entropy_beta=ENTROPY_BETA,
    use_baseline=True, seed=SEED,
)
results = train(
    agent,
    num_episodes=NUM_EPISODES,
    eval_every=EVAL_EVERY,
    eval_episodes=EVAL_GAMES,
    random_opp_fraction=RANDOM_OPP_FRAC,
    seed=SEED,
)

print(f"Win rate as X: {results['win_rates_as_x'][-1]:.1%}")
print(f"Win rate as O: {results['win_rates_as_o'][-1]:.1%}")


---
#### **5. Learning curves**


In [ ]:
checkpoints = results["eval_checkpoints"]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(checkpoints, results["win_rates_as_x"],  label="Win as X", color="tab:blue",   linewidth=1.8)
ax.plot(checkpoints, results["win_rates_as_o"],  label="Win as O", color="tab:orange",  linewidth=1.8)
ax.plot(checkpoints, results["draw_rates_as_x"], label="Draw as X", color="tab:blue",  linewidth=1, linestyle="--")
ax.axhline(0.58, color="grey", linestyle=":", linewidth=1, label="random baseline")
ax.set_xlabel("Episode"); ax.set_ylabel("Rate")
ax.set_title("REINFORCE with baseline — win/draw rate vs random agent")
ax.legend(fontsize=9); ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()


---
#### **6. Watching two trained agents play**

Both X and O use the **greedy** policy extracted from the trained agent.


In [ ]:
reinforce_policy = make_reinforce_policy(agent, greedy=True)

print("=" * 40)
print(" REINFORCE (X)  vs  REINFORCE (O)")
print("=" * 40)
play_game(env, reinforce_policy, reinforce_policy, render=True)


In [ ]:
n = 1_000
outcomes = {1: 0, -1: 0, 0: 0}
for _ in range(n):
    outcomes[_play_silent(env, reinforce_policy, reinforce_policy)] += 1

print(f"Over {n} self-play games:")
print(f"  X wins: {outcomes[1]/n:.1%}  |  O wins: {outcomes[-1]/n:.1%}  |  Draws: {outcomes[0]/n:.1%}")


---
#### **7. Monte Carlo Tree Search (PL9)**

From **model-free** (REINFORCE — learns weights from experience) to **model-based planning** (MCTS — uses the environment itself as a perfect model, plans at decision time, no training):

|  | REINFORCE (model-free) | MCTS (model-based) |
|---|---|---|
| Requires training? | ✅ yes | ❌ no — search at decision time |
| Uses environment model? | ❌ only real experience | ✅ simulates futures internally |
| Stores knowledge in | weight matrix $\theta$ | search tree (discarded after each move) |
| Compute cost | upfront (training) | per move (planning) |

Each MCTS **simulation** runs four phases:

1. **Selection** — descend with UCB1 until reaching a node that is not fully expanded or is terminal:
$$\text{UCB1}(s,a) = -Q(s,a) + c\,\sqrt{\frac{\ln N(s)}{N(s,a)}}$$
   The minus sign on $Q$ exists because parent and child are **opponents** — what is good for the child is bad for the parent.
2. **Expansion** — add one new child node for an untried action.
3. **Simulation** — random rollout from the new node to a terminal state.
4. **Backup** — propagate the outcome up the tree, negating the sign at each level (opponents alternate).

Action selection: **most-visited child** of the root (robust best — less sensitive to outlier rollouts than argmax-Q).


---
#### **8. MCTS in action — no training needed**

Unlike REINFORCE, MCTS is ready to play immediately.


In [ ]:
mcts_agent  = MCTSAgent(n_simulations=1_000)
mcts_policy = make_mcts_policy(mcts_agent)

print("=" * 40)
print(" MCTS (X)  vs  Random (O)")
print("=" * 40)
play_game(env, mcts_policy, random_policy, render=True)


---
#### **9. Effect of the simulation budget**

More simulations → deeper search → stronger play, at the cost of time per move.


In [ ]:
sim_counts = [10, 50, 100, 200, 500, 1000]
win_x, win_o, times = [], [], []
N_EVAL = 100   # keep small so the notebook stays responsive

for n in sim_counts:
    a = MCTSAgent(n_simulations=n)
    t0 = time.time()
    wx, _, _ = mcts_eval_vs_random(env, a, n_games=N_EVAL, as_player=1)
    wo, _, _ = mcts_eval_vs_random(env, a, n_games=N_EVAL, as_player=-1)
    elapsed = time.time() - t0
    win_x.append(wx); win_o.append(wo); times.append(elapsed)
    print(f"  n_sims={n:<5d}  win(X)={wx:.0%}  win(O)={wo:.0%}  ({elapsed:.1f}s)")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sim_counts, win_x, "o-", color="tab:blue",   linewidth=1.8, label="Win as X")
ax.plot(sim_counts, win_o, "s-", color="tab:orange", linewidth=1.8, label="Win as O")
ax.axhline(0.58, color="grey", linestyle=":", linewidth=1, label="random baseline")
ax.set_xscale("log")
ax.set_xlabel("Number of simulations (log scale)"); ax.set_ylabel("Win rate vs random")
ax.set_title("MCTS vs Random — effect of simulation budget")
ax.legend(fontsize=9); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()


---
#### **10. MCTS vs trained REINFORCE**

Pit a fresh (untrained) MCTS against the REINFORCE agent we trained in Section 4.


In [ ]:
mcts_strong = MCTSAgent(n_simulations=500)
N = 100

w_x, d_x, l_x = evaluate_mcts_vs_reinforce(env, mcts_strong, reinforce_policy, n_games=N, mcts_as_player=1)
w_o, d_o, l_o = evaluate_mcts_vs_reinforce(env, mcts_strong, reinforce_policy, n_games=N, mcts_as_player=-1)

print(f"MCTS (500 sims) vs REINFORCE  over {N} games per role:")
print(f"  MCTS as X: win={w_x:.0%}  draw={d_x:.0%}  loss={l_x:.0%}")
print(f"  MCTS as O: win={w_o:.0%}  draw={d_o:.0%}  loss={l_o:.0%}")


In [ ]:
labels = ["MCTS as X", "MCTS as O"]
wins   = [w_x, w_o]; draws = [d_x, d_o]; losses = [l_x, l_o]
x = np.arange(len(labels)); width = 0.25

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(x - width, wins,   width, label="Win",  color="tab:green")
ax.bar(x,         draws,  width, label="Draw", color="tab:grey")
ax.bar(x + width, losses, width, label="Loss", color="tab:red")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("Rate"); ax.set_ylim(0, 1.05)
ax.set_title("MCTS (500 sims) vs trained REINFORCE")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


---
#### **11. MCTS vs MCTS — effect of the simulation budget**

Two MCTS agents face each other. Two expected patterns:

1. **Budget matters.** A higher `n_simulations` should beat a lower one consistently — the tree is deeper and more accurate.
2. **Two strong MCTS agents draw.** Tic-Tac-Toe is a solved game whose optimal outcome is a draw; two strong planners converge on the optimal play for both sides.


In [ ]:
pairings = [(1000, 50), (1000, 200), (500, 500)]
N = 30

rows = []
for strong_n, weak_n in pairings:
    strong = MCTSAgent(n_simulations=strong_n)
    weak   = MCTSAgent(n_simulations=weak_n)
    wx, dx, lx = evaluate_mcts_vs_mcts(env, strong, weak, n_games=N, a_as_player=1)
    wo, do, lo = evaluate_mcts_vs_mcts(env, strong, weak, n_games=N, a_as_player=-1)
    rows.append((strong_n, weak_n, wx, dx, lx, wo, do, lo))
    print(f"MCTS({strong_n}) vs MCTS({weak_n})  (stronger's perspective)")
    print(f"  as X: win={wx:.0%}  draw={dx:.0%}  loss={lx:.0%}")
    print(f"  as O: win={wo:.0%}  draw={do:.0%}  loss={lo:.0%}")


In [ ]:
fig, axes = plt.subplots(1, len(rows), figsize=(4 * len(rows), 4), sharey=True)
if len(rows) == 1:
    axes = [axes]

for ax, (sn, wn, wx, dx, lx, wo, do, lo) in zip(axes, rows):
    labels = ["as X", "as O"]
    x = np.arange(len(labels)); width = 0.25
    ax.bar(x - width, [wx, wo], width, label="Win",  color="tab:green")
    ax.bar(x,         [dx, do], width, label="Draw", color="tab:grey")
    ax.bar(x + width, [lx, lo], width, label="Loss", color="tab:red")
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylim(0, 1.05)
    ax.set_title(f"MCTS({sn}) vs MCTS({wn})")

axes[0].set_ylabel("Rate (stronger's perspective)")
axes[-1].legend(fontsize=9, loc="upper right")
plt.suptitle("MCTS vs MCTS — effect of the simulation budget", y=1.02)
plt.tight_layout()
plt.show()


---
#### **12. Summary — algorithms compared on Tic-Tac-Toe**

| Algorithm | Type | Training needed? | Win rate vs random (as X) |
|---|---|:---:|:---:|
| Random policy | Baseline | — | ~58% |
| REINFORCE + baseline | Policy gradient (model-free) | ✅ | ~96% |
| MCTS (1000 sims) | Model-based planning | ❌ | ~100% |

**Key takeaway.** Model-based planning *with a perfect simulator* beats model-free learning on a small fully-observable game without needing any training — but each move costs a full search. Model-free methods amortise that cost upfront into weights.


---
#### **13. Play against MCTS**

```
 0 | 1 | 2
-----------
 3 | 4 | 5
-----------
 6 | 7 | 8
```

Set `human_starts=True` to play as X (first), `False` to play as O (second).


In [ ]:
play_game_vs_human(env, mcts_policy, human_starts=False, render=True)
